In [1]:
#%%
import duckdb
# Connect to DuckDB
con = duckdb.connect()
csv_path = "data/raw/master_clauses.csv"
# %%
num_contracts = con.execute(f"SELECT COUNT(*) FROM '{csv_path}'").fetchone()[0]

In [3]:
columns = con.execute(f"DESCRIBE SELECT * FROM '{csv_path}'").fetchall()
num_categories = len(columns) - 1

print(f"Number of Contracts: {num_contracts}")
print(f"Number of Label Categories: {num_categories}")

Number of Contracts: 510
Number of Label Categories: 82


In [5]:
import pandas as pd

In [8]:
df = con.execute(f"SELECT * FROM '{csv_path}'").df()
feature_cols = df.columns[1:]
non_empty_shares = {}
for col in feature_cols:
    non_empty_count = df[col].apply(lambda x: pd.notna(x) and str(x).strip() != "").sum()
    non_empty_shares[col] = non_empty_count / len(df)

In [11]:
shares_df = pd.DataFrame(list(non_empty_shares.items()), columns=['Category', 'Non_Empty_Share'])
shares_df = shares_df.sort_values(by='Non_Empty_Share', ascending=False)
shares_df.head(10)


,Category,Non_Empty_Share
0,Document Name,1.0
61,Irrevocable Or Perpetual License-Answer,1.0
59,Unlimited/All-You-Can-Eat-License-Answer,1.0
58,Unlimited/All-You-Can-Eat-License,1.0
57,Affiliate License-Licensee-Answer,1.0
56,Affiliate License-Licensee,1.0
55,Affiliate License-Licensor-Answer,1.0
54,Affiliate License-Licensor,1.0
53,Non-Transferable License-Answer,1.0
52,Non-Transferable License,1.0


In [13]:
import os

# Find a contract that has non-empty values for Parties and Governing Law
sample_row = con.execute(
    f'SELECT * FROM "{csv_path}" WHERE "Governing Law" != \'\' AND "Parties" != \'\' LIMIT 1'
).fetchdf().iloc[0]

doc_name = sample_row.iloc[0]
governing_law = sample_row['Governing Law']
parties = sample_row['Parties']
effective_date = sample_row['Effective Date']

print(f"Selected Document: {doc_name}\n")
print(f"Label 1 (Parties): {parties}")
print(f"Label 2 (Governing Law): {governing_law}")
print(f"Label 3 (Effective Date): {effective_date}\n")

# Open the corresponding TXT file from your file tree
txt_path = os.path.join("data/raw/full_contract_txt", doc_name)

if os.path.exists(txt_path):
    with open(txt_path, 'r', encoding='utf-8', errors='ignore') as f:
        text = f.read()
    
    # Verify by checking if the strings exist in the raw text
    print("--- Verification ---")
    print(f"Parties found in text? {'Yes' if parties in text else 'No'}")
    print(f"Governing Law found in text? {'Yes' if governing_law in text else 'No'}")
    print(f"Effective Date found in text? {'Yes' if effective_date in text else 'No'}")
else:
    print(f"Could not find text file at: {txt_path}")